In [3]:
import pandas as pd
from gurobipy import *

# Data
data2 = pd.read_csv("/Users/linus/PycharmProjects/scientificProject_operation_research_class/data/data_portfolio.csv")
stocks = data2.columns.values # Get the column names (stock symbols)

ret_mean = data2.mean() # Calculate the mean returns for each stock
ret_mean = dict(ret_mean) # Convert the mean returns to a dictionary
ret_cov = data2.cov() # Calculate the covariance matrix of the returns

budget = 1000 # Set the budget for investment
exp_ret = 5 # Set the expected return threshold

# Model
m = Model("Portfolio_max_probability") # Create a new optimization model
x_ = m.addVars(stocks, lb=0, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name='x/y') # Create decision variables for x/y
z = m.addVar(lb=0, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name='1/y') # Create a decision variable for 1/y
m.setObjective(quicksum(ret_cov.loc[i,j]*x_[i]*x_[j] for i in stocks for j in stocks), sense=GRB.MINIMIZE) # Set the objective function to maximize the probability of the expected return
m.addConstr(x_.sum() <= budget*z, name='budget_con') # Add a constraint for the budget
m.addConstr(x_.prod(ret_mean) - exp_ret*z == 1, name="µ'*x_-exp_ret*z=1") # Add a constraint for the expected return
m.write('model2.lp') # Write the model to a file
m.optimize() # Solve the optimization problem

# Result
if m.status == GRB.OPTIMAL: # Check if an optimal solution is found
    if z.x > 0: # Check if z.x is greater than 0 to avoid division by zero
        for v in m.getVars(): # Iterate through the decision variables
            if v.varName != '1/y': # Exclude the variable for 1/y
                print('x[%s]: %g' % (v.varName.split('[')[-1].replace(']', ''), v.x / z.x)) # Print the stock investment in the desired format
        print('portfolio min objective value: %g' % m.objVal) # Print the objective value

        stock_invested = [s for s in stocks if x_[s].x / z.x > 1e-3] # Get the stocks that have a positive investment
        print('stocks invested: %s' % stock_invested) # Print the stocks that are invested in
    else:
        print('z.x is zero, cannot divide by zero.') # Handle the case where z.x is zero
else:
    print('No solution') # Print a message if no solution is found


Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

Optimize a model with 2 rows, 6 columns and 12 nonzeros
Model fingerprint: 0xbcab486f
Model has 15 quadratic objective terms
Coefficient statistics:
  Matrix range     [3e-03, 1e+03]
  Objective range  [0e+00, 0e+00]
  QObjective range [1e-04, 4e-03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 1 rows and 1 columns
Presolve time: 0.00s
Presolved: 1 rows, 5 columns, 5 nonzeros
Presolved model has 15 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 4
 AA' NZ     : 1.000e+01
 Factor NZ  : 1.500e+01
 Factor Ops : 5.500e+01 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl     Time
   0   6.61788207e+03 -6.61788207e+